# Acquire public order books: OKX and Bitfinex

**Question:** what visible liquidity can I record now and reconstruct later?
This notebook records the public Bitfinex R0 and OKX books feeds for bounded
20-second samples, then downloads the configured OKX historical L2 archive.
No Binance or Coinbase connection, account, credential file or API key is used.
Public endpoints can still deny access or become unavailable; failures stay visible.

The sources are order books, not trades. Bitfinex uses BTC/USD and individual visible
orders; OKX uses BTC-USDT and aggregate price levels. Captures run sequentially, so
their intervals are not simultaneous. There is no cross-venue return or fill claim.
After acquisition, use the printed directories in
[research-foundations.ipynb](research-foundations.ipynb) for network-free inspection.

Restart the kernel and Run All. To reuse an existing historical archive while making
new live captures, set `ALPHA_OKX_ARCHIVE_DIRECTORY` to its printed directory.
Without that override, this notebook downloads a new archive (about 104 MiB for
the maintained day, plus retained working bytes).


## 1. Which source and implementation will this study use?

The setup selects the maintained venue configurations and identifies the installed
code and lockfile. **Read:** these are the inputs for this run, not a comparison of
venue quality. Configurations bound recording duration, message/storage size and
visible state. [Setup help](notebook-operations.md#before-executing).


In [1]:
# standard library imports
import hashlib
import json
import os
import shutil
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from pathlib import Path
from tempfile import TemporaryDirectory

# third party library imports
import pandas as pd

# project library imports
import alpha_research
from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.live.transport import VenueTransport
from alpha_research.data.venues.okx.book import OKXBook
from alpha_research.data.venues.okx.history_config import OKXArchiveRequest, OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.pipelines.acquisition import acquire_history
from alpha_research.pipelines.live import InstalledIdentity, collect
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/live").is_dir(), "Run from the repository root"
assert sys.version.split()[0] == (root / ".python-version").read_text().strip()
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts
for path in (root / "src/alpha_research").rglob("*"):
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}:
        assert (
            path.read_bytes()
            == (installed / path.relative_to(root / "src/alpha_research")).read_bytes()
        )
codec = DomainCodec()
implementation = InstalledIdentity.sha256()
bitfinex = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
okx = CaptureConfig.read(root / "configs/live/okx-btc-usdt.json")
okx_history = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
assert okx.instrument_id == okx_history.instrument_id
pd.DataFrame(
    [
        {
            "python": sys.version.split()[0],
            "package": alpha_research.__file__,
            "implementation_sha256": implementation,
            "lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
        }
    ]
)

,python,package,implementation_sha256,lock_sha256
0,3.14.7,/private/tmp/mr27-validation/installed/venv/li...,26df28cd5d421486c0f3fbf9a299ea326de5c2730d3c6a...,e6cd3a169ff4b6850bcb5b09e38c44e1179b0dc1fb7c6a...


## 2. Which instruments and types of liquidity can I observe?

**Read:** the discovery table describes current source availability and the selected
native instruments. It does not prove historical listing membership. Bitfinex R0
reports visible individual orders. OKX public `books` reports 400 aggregate price
levels per side at its configured nominal update cadence.

Keep the representation and quote currency attached to any comparison: aggregate
price levels are not individual orders, and BTC/USD is not BTC/USDT.


In [2]:
discovery = []
for config in (bitfinex, okx):
    payload = VenueTransport(config).discover()
    discovery.append(
        {
            "venue": config.venue,
            "representation": "L2" if config.venue == "okx" else "L3",
            "native_symbol": config.symbol,
            "instrument_id": str(config.instrument_id),
            "channel": config.channel,
            "duration_s": config.duration_seconds,
            "message_limit": config.max_messages,
            "journal_bytes": config.max_bytes,
            "depth": config.depth,
            "discovery_sha256": hashlib.sha256(payload).hexdigest(),
        }
    )
pd.DataFrame(discovery)

,venue,representation,native_symbol,instrument_id,channel,duration_s,message_limit,journal_bytes,depth,discovery_sha256
0,bitfinex,L3,tBTCUSD,c87d983f-e7ec-4e64-8d54-9f2be67a64b8,R0,20,10000,32000000,25,ad5c2c16d99b75060bdfcb43aa70faa233931623d8b953...
1,okx,L2,BTC-USDT,4476df8e-6342-4417-839e-e41b88f41446,books,20,10000,32000000,400,5868aba36f44605b113b2602986f4e2b2e08c55b6bb638...


## 3. What happened during this bounded live sample?

The next cell records Bitfinex and OKX, saves the incoming messages and reconstructs
the visible book. **Read:** the returned status, message count, usable interval and
gap reasons describe this particular attempt. A later successful recovery does not
erase a preceding invalid interval.

Each call stops at configured limits and retains its own recording directory.
Connection/control messages and local receipt times stay with market messages,
so later inspection can explain why a book state was or was not usable.


In [3]:
bitfinex_snapshots = root / "data/bitfinex/book-snapshots"
okx_snapshots = root / "data/okx/live/book-snapshots"
for destination in (bitfinex_snapshots, okx_snapshots):
    destination.mkdir(parents=True, exist_ok=True)
bitfinex_result = await collect(
    bitfinex,
    root / "data/bitfinex",
    implementation_identity=implementation,
    snapshot_directory=bitfinex_snapshots,
    snapshot_records=32,
)
assert bitfinex_result["replay_parity"] and bitfinex_result["canonical_parity"]
assert bitfinex_result["synchronized"] and not bitfinex_result["gaps"], bitfinex_result
okx_result = await collect(
    okx,
    root / "data/okx/live",
    implementation_identity=implementation,
    snapshot_directory=okx_snapshots,
    snapshot_records=32,
)
assert okx_result["replay_parity"] and okx_result["canonical_parity"]
assert okx_result["synchronized"] and not okx_result["gaps"], okx_result
assert okx_result["orders"] is None and okx_result["representation"] == "L2"
{"bitfinex": bitfinex_result, "okx": okx_result}

{'bitfinex': {'venue': 'bitfinex',
  'symbol': 'tBTCUSD',
  'instrument_id': 'c87d983f-e7ec-4e64-8d54-9f2be67a64b8',
  'channel': 'R0',
  'started_ns': 1789801990636491000,
  'finished_ns': 1789802010852935000,
  'messages': 315,
  'frames': 315,
  'duplicates': 0,
  'stale_snapshots': 0,
  'retry_exhausted': False,
  'updates': 309,
  'snapshots': 1,
  'heartbeats': 2,
  'depth': 25,
  'orders': 50,
  'max_orders': 100000,
  'gaps': [],
  'synchronized': True,
  'ended': True,
  'source_sequence': 312,
  'last_source_time': None,
  'book_identity': 'c030a2860e130cbbf2a4c6db862409413e2635ff2d8cee0e37d49e77ecb8bbbb',
  'trace_identity': '94434b939dff1ae34c5be0e8532a90e5a17e4a277ef7af8ef53d20d4e0cce234',
  'epochs': 1,
  'quality': {'status': 'unavailable',
   'reason': 'duration_limit',
   'scope': 'bitfinex:R0:c87d983f-e7ec-4e64-8d54-9f2be67a64b8',
   'transition_ns': 1789802010850748000,
   'last_durable_ns': 1789802010852935000,
   'last_market_ns': 1789802010527261000,
   'watermark

## 4. Which interval and part of the book does my sample cover?

**Read:** counts and identities refer to the actual recording. Local receipt clocks
are UTC observations by this application, not invented exchange timestamps.
Bitfinex has no source clock under the selected settings; its depth-25 window is a
bounded visible sample. OKX reports aggregate levels, with individual `orders=None`.

Do not compare the order and level counts as the same quantity. An unknown clock
or count remains unknown, rather than being replaced with zero.


In [4]:
results = [bitfinex_result, okx_result]
rows = []
for result in results:
    row = {
        name: result[name]
        for name in (
            "venue",
            "symbol",
            "messages",
            "snapshots",
            "updates",
            "orders",
            "depth",
            "gaps",
            "saved_input_identity",
        )
    }
    row["levels"] = result.get("levels")
    row["representation"] = "L2" if result["venue"] == "okx" else "L3"
    for name in ("started_ns", "finished_ns"):
        row[name.replace("_ns", "_utc")] = (
            datetime(1970, 1, 1, tzinfo=UTC) + timedelta(microseconds=result[name] // 1000)
        ).isoformat()
    rows.append(row)
pd.DataFrame(rows)

,venue,symbol,messages,snapshots,updates,orders,depth,gaps,saved_input_identity,levels,representation,started_utc,finished_utc
0,bitfinex,tBTCUSD,315,1,309,50.0,25,[],f249e9e5566412ff4e8b33190ebd5133928a6b548cae3b...,NaN,L3,2026-09-19T07:13:10.636491+00:00,2026-09-19T07:13:30.852935+00:00
1,okx,BTC-USDT,156,1,154,NaN,400,[],0de8f9c9e6f8b56789b509e2f0402cb982d8641cd9f218...,800.0,L2,2026-09-19T07:13:30.886292+00:00,2026-09-19T07:13:51.008629+00:00


## 5. Can I trace the reconstruction back to the saved messages?

The next cell verifies saved bytes and inspects the receipt record and numeric
types used by the same decoder as collection. **Read:** the fingerprint links the
reconstruction to an original file; exact numeric representation preserves prices
and sizes. Bulk feed data remain in the ignored venue directory.

For OKX, an absolute size replaces the prior size at a price and zero deletes that
level. Live sequence links help detect invalid transitions. These mechanics support
reconstruction; they are not assumptions about a strategy's fills.


In [5]:
capture_root = Path(bitfinex_result["capture_directory"])
request = codec.decode_json((capture_root / "request.json").read_bytes(), CaptureRequest)
journal = CaptureJournal(capture_root, request)
first_frame = next(record for record in journal.records() if record.kind == "frame")
offline = CaptureReplay.read(capture_root)
example_order = next(iter(offline.book.orders.values()))
okx_root = Path(okx_result["capture_directory"])
okx_offline = CaptureReplay.read(okx_root)
assert isinstance(okx_offline.book, OKXBook)
okx_price, okx_level = next(iter(okx_offline.book.bids.items()))
pd.DataFrame(
    [
        {
            "venue": "bitfinex",
            "epoch": first_frame.epoch,
            "receipt_ns": first_frame.received_ns,
            "raw_frame_bytes": len(first_frame.raw),
            "raw_frame_sha256": hashlib.sha256(first_frame.raw).hexdigest(),
            "order_id_type": type(example_order.order_id).__name__,
            "price_type": type(example_order.price).__name__,
            "size_type": type(example_order.size).__name__,
            "source_time": offline.book.source_time,
        },
        {
            "venue": "okx",
            "order_id_type": None,
            "price_type": type(okx_price).__name__,
            "size_type": type(okx_level.size).__name__,
            "reported_orders_at_price": okx_level.order_count,
            "source_time": okx_offline.book.source_time,
            "source_time_unit": "unix_milliseconds",
        },
    ]
)

,venue,epoch,receipt_ns,raw_frame_bytes,raw_frame_sha256,order_id_type,price_type,size_type,source_time,reported_orders_at_price,source_time_unit
0,bitfinex,0.0,1.789802e+18,102.0,87fae0ea3d8f273d7340b583d3415181b8f9dea2e84624...,str,Decimal,Decimal,NaN,NaN,NaN
1,okx,NaN,NaN,NaN,NaN,NaN,Decimal,Decimal,1789802030809,1.0,unix_milliseconds


## 6. Does the reconstructed book survive a data gap?

First compare the saved reconstruction with the recording's sequence of book
states. **Read:** matching results establish repeatable processing of these saved
messages. Keep the actual files, request and manifest; a matching fingerprint
cannot recover a lost file.

**Synthetic challenge:** append a disconnect to a copy of the observed prefix.
A heartbeat alone must not make the book usable again: a fresh subscription and
snapshot are required. The real saved input is unchanged. A recovered book does
not retroactively fill the missing interval.


In [6]:
assert offline.summary()["trace_identity"] == bitfinex_result["trace_identity"]
assert offline.summary()["book_identity"] == bitfinex_result["book_identity"]
assert offline.messages == bitfinex_result["messages"]
assert okx_offline.summary()["trace_identity"] == okx_result["trace_identity"]
assert okx_offline.summary()["book_identity"] == okx_result["book_identity"]
exercise = CaptureReplay(bitfinex)
for record in journal.records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(
    CaptureRecord(
        kind="gap",
        epoch=exercise.epoch,
        received_ns=exercise.finished_ns,
        reason="synthetic_disconnect",
    )
)
assert not exercise.book.ready
{
    "offline_parity": True,
    "okx_offline_parity": True,
    "verified_messages": offline.messages,
    "trace_sha256": offline.summary()["trace_identity"],
    "synthetic_gap": exercise.gaps[-1],
    "usable_after_gap": exercise.book.ready,
}

{'offline_parity': True,
 'okx_offline_parity': True,
 'verified_messages': 315,
 'trace_sha256': '94434b939dff1ae34c5be0e8532a90e5a17e4a277ef7af8ef53d20d4e0cce234',
 'synthetic_gap': {'epoch': 0,
  'received_ns': 1789802010850748000,
  'reason': 'synthetic_disconnect'},
 'usable_after_gap': False}

## 7. What does the selected historical OKX archive contain?

The maintained request selects **8 September 2026**, one native instrument and the
400-level L2 archive. Without a path override, the next cell discovers the source
reference, downloads the compressed day, verifies it and saves the original bytes
with its request. With `ALPHA_OKX_ARCHIVE_DIRECTORY`, it verifies and replays that
saved archive instead. The printed directory and receipt time distinguish the two.
A completed download establishes a retained archive, not full-day continuity or
an accepted research dataset.

The original recorded download was about 104 MiB. Configured caps are 160 MB
compressed and 1 GB expanded. An incomplete attempt has no completion manifest;
retrying acquisition starts its download again within finite limits.
[Acquisition details](../docs/data/okx-market-data.md).


In [7]:
archive_override = os.environ.get("ALPHA_OKX_ARCHIVE_DIRECTORY")
if archive_override:
    archive_directory = Path(archive_override).expanduser().resolve()
    saved_request = codec.decode_json(
        (archive_directory / "request.json").read_bytes(), OKXArchiveRequest
    )
    if saved_request.config != okx_history:
        raise ValueError("Selected archive differs from the maintained OKX configuration")
    history_result = OKXArchiveStore(archive_directory).replay()
else:
    history_result = acquire_history(
        okx_history, root / "data/okx/history", implementation_identity=implementation
    )
assert history_result["state_available"]
history_result

{'venue': 'okx',
 'symbol': 'BTC-USDT',
 'instrument_id': '4476df8e-6342-4417-839e-e41b88f41446',
 'representation': 'L2',
 'mode': 'historical',
 'depth': 400,
 'orders': None,
 'records': 10000,
 'snapshots': 1,
 'updates': 9999,
 'levels': 800,
 'source_sequence': None,
 'first_source_ms': '1788825600004',
 'last_source_ms': '1788825824624',
 'record_receipt_time': None,
 'continuity': 'unproven_no_source_sequences',
 'gaps': None,
 'state_available': True,
 'expanded_bytes': 764149760,
 'stop_reason': 'record_limit',
 'entire_archive_replayed': False,
 'book_identity': '98463d3904ce63326fb0e2a9bab95b927a6068289aa1989cebe66e18fabdf315',
 'trace_identity': 'e8c58ef9bc585227184ea43f5ced9ded45ba451ea91b0251544056ef5c5e6811',
 'saved_input_identity': 'fef59b343d1972c32f67583edc538c5d2f7350e6f3a0059e6f87896d7e8bf676',
 'saved_bytes': 109423187,
 'request_identity': '71672efd00a8f3f6c959d0a08f8ce2a8793b3fb84d23f1a3c22b2756a6ec2665',
 'index_identity': '288481f38d44b22f6b8bd4b1923894fa40db

## 8. How much historical book state did I actually reconstruct?

The whole compressed archive is retained, but this notebook processes only the
first **10,000 records**. **Read:** the displayed source interval describes that
prefix, not a full day of interpreted data.

The historical records provide price, size and order-count fields but no original
receipt timestamps or sequence IDs. Therefore `gaps=None` means continuity is
unknown; it does not mean zero gaps. These records cannot automatically repair
live gaps, and archive sampling is not assumed equivalent to the live feed.


In [8]:
offline_history = OKXArchiveStore(Path(history_result["archive_directory"])).replay()
assert offline_history["trace_identity"] == history_result["trace_identity"]
assert offline_history["book_identity"] == history_result["book_identity"]
assert offline_history["records"] == history_result["records"]
assert offline_history["source_sequence"] is None and offline_history["gaps"] is None
pd.DataFrame(
    [
        {
            key: offline_history[key]
            for key in (
                "venue",
                "symbol",
                "records",
                "snapshots",
                "updates",
                "levels",
                "depth",
                "first_source_ms",
                "last_source_ms",
                "continuity",
                "entire_archive_replayed",
                "stop_reason",
                "saved_input_identity",
                "trace_identity",
            )
        }
    ]
)

,venue,symbol,records,snapshots,updates,levels,depth,first_source_ms,last_source_ms,continuity,entire_archive_replayed,stop_reason,saved_input_identity,trace_identity
0,okx,BTC-USDT,10000,1,9999,800,400,1788825600004,1788825824624,unproven_no_source_sequences,False,record_limit,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...,e8c58ef9bc585227184ea43f5ced9ded45ba451ea91b02...


## 9. What can I compare across these sources?

Read source representation, observed interval, clocks, saved identity and coverage
together. Receipt intervals refer to live recordings; source intervals refer to
the processed historical prefix. These are different intervals and USD/USDT are
different quote currencies. Differences in these columns limit comparisons of
apparent activity or liquidity across sources.


In [9]:
epoch = datetime(1970, 1, 1, tzinfo=UTC)
evidence = []
for result in results:
    evidence.append(
        {
            "venue": result["venue"],
            "symbol": result["symbol"],
            "mode": "live",
            "representation": "L2" if result["venue"] == "okx" else "L3",
            "start_utc": (epoch + timedelta(microseconds=result["started_ns"] // 1000)).isoformat(),
            "end_utc": (epoch + timedelta(microseconds=result["finished_ns"] // 1000)).isoformat(),
            "clock": "receipt",
            "records": result["messages"],
            "orders": result["orders"],
            "levels": result.get("levels"),
            "depth": result["depth"],
            "gaps": result["gaps"],
            "saved_input_sha256": result["saved_input_identity"],
        }
    )
evidence.append(
    {
        "venue": "okx",
        "symbol": okx_history.symbol,
        "mode": "historical prefix",
        "representation": "L2",
        "clock": "source",
        "records": offline_history["records"],
        "start_utc": (
            epoch + timedelta(milliseconds=int(offline_history["first_source_ms"]))
        ).isoformat(),
        "end_utc": (
            epoch + timedelta(milliseconds=int(offline_history["last_source_ms"]))
        ).isoformat(),
        "orders": None,
        "levels": offline_history["levels"],
        "depth": 400,
        "gaps": None,
        "saved_input_sha256": offline_history["saved_input_identity"],
    }
)
pd.DataFrame(evidence)

,venue,symbol,mode,representation,start_utc,end_utc,clock,records,orders,levels,depth,gaps,saved_input_sha256
0,bitfinex,tBTCUSD,live,L3,2026-09-19T07:13:10.636491+00:00,2026-09-19T07:13:30.852935+00:00,receipt,315,50.0,NaN,25,[],f249e9e5566412ff4e8b33190ebd5133928a6b548cae3b...
1,okx,BTC-USDT,live,L2,2026-09-19T07:13:30.886292+00:00,2026-09-19T07:13:51.008629+00:00,receipt,156,NaN,800.0,400,[],0de8f9c9e6f8b56789b509e2f0402cb982d8641cd9f218...
2,okx,BTC-USDT,historical prefix,L2,2026-09-08T00:00:00.004000+00:00,2026-09-08T00:03:44.624000+00:00,source,10000,NaN,800.0,400,None,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...


## 10. Can published capture feed the shared consumer and resume? (MR 27)

The live collector retained and compared the first 32 whole message boundaries
before publishing each book snapshot. Now reopen those exact inputs through the
shared query/stepping path and compare the same visible-spread consumer. A fresh
process restores the accepted snapshot and recovery epoch with its shared cursor.
The OKX archive is inspected separately: its original availability remains unknown.


In [10]:
with TemporaryDirectory(prefix="book-parity-capture-") as temporary:
    check_root = Path(temporary)
    script = check_root / "parity_consumer.py"
    shutil.copyfile(root / "tests/integration/consumers/capture/parity_consumer.py", script)
    check = subprocess.run(
        [
            sys.executable,
            "-I",
            str(script),
            str(check_root / "evidence"),
            bitfinex_result["capture_directory"],
            okx_result["capture_directory"],
            history_result["archive_directory"],
        ],
        cwd=check_root,
        capture_output=True,
        text=True,
        check=True,
        timeout=180,
    )
    integrated = json.loads(check.stdout)
assert all(row["live_prefix_compared"] for row in integrated["inputs"][:2])
assert integrated["inputs"][2]["scheduled_records"] == 0
pd.DataFrame(
    [
        {
            key: row[key]
            for key in (
                "source",
                "raw_identity",
                "snapshot_identity",
                "replay_identity",
                "query_records",
                "scheduled_records",
                "completed_checkpoints",
                "fresh_process_resume",
                "live_prefix_compared",
            )
        }
        for row in integrated["inputs"]
    ]
)

,source,raw_identity,snapshot_identity,replay_identity,query_records,scheduled_records,completed_checkpoints,fresh_process_resume,live_prefix_compared
0,/Users/lxg/Documents/GitHub/Alpha Research/alp...,f249e9e5566412ff4e8b33190ebd5133928a6b548cae3b...,0bd0f3424bd8cf05b10e9b9b99fa8d5178e5accf69d54d...,9bcd9ccb63b7732618f66afd5be81e26c0e7bc0bfe760c...,32,32,33,True,True
1,/Users/lxg/Documents/GitHub/Alpha Research/alp...,0de8f9c9e6f8b56789b509e2f0402cb982d8641cd9f218...,ba39e623f28911a0cc5959452f88e0f43b7abc5664f0e2...,6123a25ab2b9169eb6711630631dd8f86cee99c46f90f4...,32,32,33,True,True
2,/Users/lxg/Documents/GitHub/Alpha Research/alp...,fef59b343d1972c32f67583edc538c5d2f7350e6f3a005...,794d083f7e7eaf13cd88a0839b1bbc39139b2b4b0f91b5...,ae583a279a0e0558297108ad70680a19c0b0c3016b9e3c...,32,0,1,True,False


The live prefix, independent raw replay, published canonical events and consumer
outputs agree. Canonical comparison covers the selected prefix; the older native
summary comparison covers the entire finite capture. A mismatch retains
`first-divergence.json` with the raw position and both observations. Exact receipts
remain nanoseconds; shared clock steps round up to microseconds. Query batch size
never splits a source message or changes the completed cursor identity.

Full depth, recorder completeness, Coinbase live access and executable consumers
remain unproven. These results complete the named-source MR 27 checkpoint for
bounded exploratory use; they do not claim all of M3 or deploy a collector.


## What this study establishes, and where to stop

The outputs identify both recorded intervals, source depth, gaps and saved inputs.
Matching replay establishes reconstruction from those bytes. Historical OKX
continuity remains unknown; processing a prefix does not establish full-day coverage.
Neither L2 nor visible R0 orders establish hidden liquidity, queue priority or fills.

Copy the `archive_directory` into `ALPHA_OKX_ARCHIVE_DIRECTORY` and Bitfinex's
`capture_directory` into `ALPHA_BITFINEX_CAPTURE_DIRECTORY` for the
[saved-book research notebook](research-foundations.ipynb). Original bytes remain
under ignored `data/okx/` and `data/bitfinex/`; preserve them with their request and
manifest. The [guide](README.md) explains the research limits and next roadmap MR.
